In [1]:
# here we are creating a short-term memory where persistance is created by postgres database. (not InMemorySaver)

In [27]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_groq import ChatGroq
from psycopg_pool import ConnectionPool
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

In [21]:
load_dotenv()

True

In [22]:
LLM = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [23]:
# connecting DB
DB_URI = os.environ["DATABASE_URL"]

connection_kwargs = {
    "autocommit":True,
    "prepare_threshold":0
}

In [24]:
pool = ConnectionPool(
    conninfo=DB_URI,
    max_size=20,
    kwargs=connection_kwargs
)

checkpointer = PostgresSaver(pool)
checkpointer.setup()

In [25]:
def call_model(state:MessagesState):
    response = LLM.invoke(state["messages"])
    return {"messages":[response]}

In [26]:
graph_builder = StateGraph(MessagesState)

graph_builder.add_node("call_model",call_model)

graph_builder.add_edge(START,"call_model")
graph_builder.add_edge("call_model",END)

graph = graph_builder.compile(checkpointer=checkpointer)

In [30]:
thread_id = "101"

graph.invoke({"messages":[HumanMessage(content="i am bob")]},config={"configurable":{"thread_id":thread_id}})

{'messages': [HumanMessage(content='i am bob', additional_kwargs={}, response_metadata={}, id='b098a9d5-830a-426a-a639-3d021d5e28b2'),
  AIMessage(content="Hello Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 38, 'total_tokens': 64, 'completion_time': 0.062145885, 'completion_tokens_details': None, 'prompt_time': 0.005413971, 'prompt_tokens_details': None, 'queue_time': 0.051172489, 'total_time': 0.067559856}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdb90-b248-7ba0-992f-6c033e3d4535-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 26, 'total_tokens': 64})]}

In [31]:
graph.invoke({"messages":[HumanMessage(content="what is my name?")]},config={"configurable":{"thread_id":thread_id}})

{'messages': [HumanMessage(content='i am bob', additional_kwargs={}, response_metadata={}, id='b098a9d5-830a-426a-a639-3d021d5e28b2'),
  AIMessage(content="Hello Bob! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 38, 'total_tokens': 64, 'completion_time': 0.062145885, 'completion_tokens_details': None, 'prompt_time': 0.005413971, 'prompt_tokens_details': None, 'queue_time': 0.051172489, 'total_time': 0.067559856}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdb90-b248-7ba0-992f-6c033e3d4535-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 26, 'total_tokens': 64}),
  HumanMessage(content='what is my name?', additional_kwargs={}, response_metadata={}, id